# MGT-432 Group Assignment · Team notebook

This notebook trains your team's model on `train.csv` and the small labelled `adaptation.csv`, saves it as `team_model.pkl`, and uses the saved model to write two files:

- `predictions.csv`: a revenue forecast for every test film
- `valuations.csv`: your auction sheet, one row per auction lot with whatever columns you want to bid with

**What we do with it.** After the deadline we run section 6 with your `team_model.pkl`, using the hidden test and auction files. We score only `predicted_revenue`. We print the complete `valuations.csv` and give it to your team before the auction. If your saved model cannot be loaded on our computer, we run the whole notebook instead.

**What you do**

1. Improve the cells marked ✏️ (sections 0, 2 to 5). Leave sections 1 and 6 as they are.
2. List every library you import in `requirements.txt` (section 0).
3. Run all cells. Section 5 saves your model as `team_model.pkl` and checks `requirements.txt`; section 6 loads the model, writes the two files and prints two ✅ lines.
4. Check that section 6 works on its own: **restart the kernel** (clearing all outputs is fine), then run only the two cells of section 6. They must print the two ✅ lines again.
5. Upload three files per team: this `.ipynb`, `team_model.pkl` and `requirements.txt`.

## 0 · Files and libraries ✏️

`requirements.txt` must list every library this notebook imports, one per line, with the version you use (for example `lightgbm==4.6.0`). We install exactly these libraries to load your model. The check at the end of section 5 tells you what to add or change.

To install the listed libraries, for example on Google Colab or a new computer, remove the `#` in front of `!pip install` and run the cell.

In [9]:
# !pip install -r requirements.txt

Change the file names only if the data are somewhere else. On Google Colab, upload `train.csv`, `adaptation.csv`, `sample_features.csv` and `requirements.txt`, and remove `../` below.

In [1]:
import os
print(os.getcwd())

/Users/alegonzalez/PycharmProjects/DSB_assignment/starter_kit


In [2]:
TRAIN_FILE = "../train.csv"
ADAPTATION_FILE = "../adaptation.csv"

## 1 · Load the training data (do not edit)

Three columns of `train.csv` are not in the test and auction files: the target `revenue` and the identifiers `movie_id` and `title`. Your features cannot use them.

`adaptation.csv` contains 396 labelled, anonymous films from the same out-of-sample setting as Hidden Test B, whose distribution may differ from `train.csv`. It has the deployment features plus `adaptation_id` and `revenue`. Do not use either ID as a feature.

In [3]:
import numpy as np
import pandas as pd

train = pd.read_csv(TRAIN_FILE)
adaptation = pd.read_csv(ADAPTATION_FILE)

print(len(train), "training films")
print(len(adaptation), "adaptation films")
train.head()

3943 training films
396 adaptation films


,movie_id,title,release_year,release_quarter,runtime,budget,original_language,primary_genre,genres,n_genres,production_countries,n_companies,company_ids,n_cast,n_crew,in_collection,keyword_ids,cast_ids,director_id,revenue
0,M3ddf258d,Jumanji,1995,4,105,65000000,en,Adventure,Adventure|Fantasy|Family,3,US,3,C6dfc8237|C6bee1098|Ceafcc337,26,16,False,Kcba84b9c|K6b9ad492|K31b25f7e|Kacea562a|K7b321...,P8b29037b|Pcc7ebcea|Pfe6d4c3d|P45aad4a2|P4c455563,Pa589cb62,262797249.0
1,M862e1fd4,Heat,1995,4,170,60000000,en,Action,Action|Crime|Drama|Thriller,4,US,3,Cc2c73c36|C4afff6ec|C3e7bdc8f,65,71,False,Kc5b65b8c|Kb1aa9b34|Kd95b93ea|K443b5d4d|K00ee1...,P13753a9d|Pd9604827|Pcfb78473|Pb3cd0383|P575dda13,Paa8ef28b,187436818.0
2,M4617e480,Sudden Death,1995,4,105,35000000,en,Action,Action|Adventure|Thriller,3,US,3,Cd7a83b16|C0866a564|C79ac32e3,6,9,False,K4d2febdc|K8b8060e4|K18f7c353|K4f7f609b,P7ff40d0f|P387c2084|P972106f7|P12aa5436|P22cc2e84,Pc2c37f32,64350171.0
3,Medf94ac1,The American President,1995,4,105,62000000,en,Comedy,Comedy|Drama|Romance,3,US,2,Cf4478484|Ce4e76224,18,6,False,Kfb526a76|K2b746c3e|Kfa895068|K1355f779|Kd57fb469,P9485f066|Peb51c4ea|P977f7843|P5768eddf|Pd3b64bd6,P4bc483e6,107879496.0
4,Mfec91032,Nixon,1995,4,190,44000000,en,History,History|Drama,2,US,2,C2b01c913|C4ca4dc32,34,8,False,K2b746c3e|K3ccb5600|K85e45b97|K4385ab38|K8beac...,Pdff627c7|P7b4520e9|P387c2084|Pceb538c7|P09ae177a,Pc20617a1,13681765.0


In [6]:
print(train.columns)

Index(['movie_id', 'title', 'release_year', 'release_quarter', 'runtime',
       'budget', 'original_language', 'primary_genre', 'genres', 'n_genres',
       'production_countries', 'n_companies', 'company_ids', 'n_cast',
       'n_crew', 'in_collection', 'keyword_ids', 'cast_ids', 'director_id',
       'revenue'],
      dtype='object')


## 2 · Features ✏️

`make_features` turns a table of films into the numbers the model learns from. The same function is applied to the training, adaptation, hidden test and auction films. It must ignore targets, titles and row identifiers.

In [7]:
numeric_columns = [
        "release_year",
        "release_quarter",
        "runtime",
        "budget",
        "n_genres",
        "n_companies",
        "n_cast",
        "n_crew",
]

boolean_columns = ["in_collection"]

categorical_columns = [
    "original_language",
    "primary_genre",
    "production_countries",
    "director_id",
]

list_columns = [
    "genres",
    "company_ids",
    "keyword_ids",
    "cast_ids",
]


In [14]:
from collections import Counter

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler


def as_list(x):
    if isinstance(x, (list, tuple, set)):
        return list(x)
    if isinstance(x, str):
        return [t.strip() for t in x.split("|") if t.strip()]
    return []


class ListEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=20):
        self.min_count = min_count

    def fit(self, X, y=None):
        counts = Counter(item for items in X for item in as_list(items))
        self.classes_ = [k for k, n in counts.items() if n >= self.min_count]
        self.mlb_ = MultiLabelBinarizer(classes=self.classes_).fit([])
        return self

    def transform(self, X):
        known = set(self.classes_)
        return self.mlb_.transform([[i for i in as_list(x) if i in known] for x in X])

    def get_feature_names_out(self, input_features=None):
        return [f"{input_features[0]}_{c}" for c in self.classes_]


numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = OneHotEncoder(
    handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipe, numeric_columns),
        ("bool", "passthrough", boolean_columns),
        ("cat", categorical_pipe, categorical_columns),
    ]
    + [(f"list_{col}", ListEncoder(min_count=20), col) for col in list_columns]
)
preprocessor.set_output(transform="pandas")


def prepare(df):
    df = df.copy()
    df[boolean_columns] = df[boolean_columns].fillna(False).astype(int)
    return df

preprocessor.fit(prepare(train))

,transformers,"[('num', ...), ('bool', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [15]:
def make_features(films):
    X = preprocessor.transform(prepare(films))
    return X

make_features(train).head()

,num__release_year,num__release_quarter,num__runtime,num__budget,num__n_genres,num__n_companies,num__n_cast,num__n_crew,bool__in_collection,cat__original_language_en,...,list_cast_ids__cast_ids_Pa9346d2f,list_cast_ids__cast_ids_P6f155c36,list_cast_ids__cast_ids_Pd4551c6e,list_cast_ids__cast_ids_P7ee731d8,list_cast_ids__cast_ids_P4b27ca24,list_cast_ids__cast_ids_P58bca367,list_cast_ids__cast_ids_Pa70948d3,list_cast_ids__cast_ids_P123185a4,list_cast_ids__cast_ids_P7b34c4be,list_cast_ids__cast_ids_P8ea19000
0,-0.699145,1.257773,-0.241207,0.750030,0.348812,-0.017147,0.119511,-0.416811,0,1.0,...,0,0,0,0,0,0,0,0,0,0
1,-0.699145,1.257773,2.963567,0.629931,1.244815,-0.017147,2.127167,1.168091,0,1.0,...,0,0,0,0,0,0,0,0,0,0
2,-0.699145,1.257773,-0.241207,0.029436,0.348812,-0.017147,-0.910056,-0.618526,0,1.0,...,0,0,0,0,0,0,0,0,0,0
3,-0.699145,1.257773,-0.241207,0.677970,0.348812,-0.473968,-0.292316,-0.704975,0,1.0,...,0,0,0,0,0,0,0,0,0,0
4,-0.699145,1.257773,3.949652,0.245614,-0.547191,-0.473968,0.531338,-0.647342,0,1.0,...,0,0,0,0,0,0,0,0,0,0


## 3 · Model ✏️

The model learns `log10(revenue)`: revenues range from thousands to billions of dollars, and the score (log-MAE) measures errors on this log scale. Fix `random_state` so that the model is the same every time you run the notebook.

In [ ]:
def make_model():
    return ...

## 4 · Validate ✏️

The hidden test has two parts, scored separately: Hidden Test A comes from the same distribution as `train.csv`; Hidden Test B is out-of-sample, and its distribution may differ. The 396 labelled adaptation films come from the same setting as Hidden Test B. The simple check below holds out about 20% of them, trains on `train.csv` plus the other 80%, and evaluates on the held-out adaptation films. You may replace it with cross-validation.

In [ ]:
def log_mae(actual, forecast):
    return np.mean(np.abs(np.log10(forecast) - np.log10(actual)))

adapt_valid = adaptation.groupby("release_year", group_keys=False).sample(frac=0.20, random_state=432)
adapt_fit = adaptation.drop(index=adapt_valid.index)
development = pd.concat([train, adapt_fit], ignore_index=True, sort=False)

check_model = make_model().fit(make_features(development), np.log10(development["revenue"]))
forecast = 10 ** check_model.predict(make_features(adapt_valid))
median_guess = np.full(len(adapt_valid), development["revenue"].median())

print(f"Trained on {len(train)} training + {len(adapt_fit)} adaptation films; checked on {len(adapt_valid)} held-out adaptation films")
print(f"  your model          log-MAE = {log_mae(adapt_valid['revenue'], forecast):.3f}")
print(f"  training median     log-MAE = {log_mae(adapt_valid['revenue'], median_guess):.3f}  (lower is better)")

## 5 · Train and save the final model ✏️

After validation, train on all training and adaptation films. Then define two functions. Each returns one result per film, in the same row order as `films`:

- `predict_revenue(films)` returns one positive revenue forecast in USD per film. This is what we score on the hidden test.
- `auction_sheet(films)` returns a DataFrame with whatever columns you want on your printed auction sheet, for example a valuation, an 80% interval, a maximum bid or a short note. It is not scored. Keep it concise so that the printout stays readable.

In [ ]:
final_train = pd.concat([train, adaptation], ignore_index=True, sort=False)
model = make_model().fit(make_features(final_train), np.log10(final_train["revenue"]))

def predict_revenue(films):
    return 10 ** model.predict(make_features(films))

def auction_sheet(films):
    return pd.DataFrame({"valuation": predict_revenue(films)})

The next two cells save `predict_revenue` and `auction_sheet` in `team_model.pkl`, together with everything they use (your model, `make_features` and any lookup tables), and check that `requirements.txt` lists every library you use. Run them again whenever you change sections 2 to 5. Do not edit them.

In [ ]:
#Do not edit it
import gzip, os, pickle, platform, sklearn
try:
    import cloudpickle                          # listed directly in requirements.txt
except ImportError:
    from joblib.externals import cloudpickle   # fallback for an older starter environment

with gzip.open("team_model.pkl", "wb") as f:
    pickle.dump({"python": platform.python_version(), "scikit-learn": sklearn.__version__}, f)
    cloudpickle.dump({"predict_revenue": predict_revenue, "auction_sheet": auction_sheet}, f)
print(f"✅ saved team_model.pkl ({os.path.getsize('team_model.pkl') / 1e6:.1f} MB)")

In [ ]:
#Do not edit it
import importlib.metadata, re, types

listed = {}
try:
    for line in open("requirements.txt"):
        line = line.split("#")[0].strip()
        if line:
            name = re.split(r"[<>=!~;\[ ]", line)[0].lower().replace("_", "-")
            listed[name] = line.split("==")[1].strip() if "==" in line else ""
except FileNotFoundError:
    print("⚠ requirements.txt not found: put it next to this notebook")

dist_of = importlib.metadata.packages_distributions()
used = set()
for obj in list(globals().values()):
    module = obj.__name__ if isinstance(obj, types.ModuleType) else getattr(obj, "__module__", None)
    if isinstance(module, str) and len(dist_of.get(module.split(".")[0], [])) == 1:
        used.add(dist_of[module.split(".")[0]][0])

missing = []
for dist in sorted(used):
    name, version = dist.lower().replace("_", "-"), importlib.metadata.version(dist)
    if name not in {"ipython", "ipykernel"} and listed.get(name) != version:
        missing.append(f"{name}=={version}")
if missing:
    print("⚠ Update requirements.txt so that it contains these lines:", *missing, sep="\n    ")
else:
    print("✅ requirements.txt lists every library you use")

## 6 · Write the output files (do not edit)

This section needs nothing from the cells above, only `team_model.pkl`. It writes `predictions.csv` from `predict_revenue` and `valuations.csv` from `auction_sheet`.

Here both output files are written from `sample_features.csv`, which has the same feature columns as the hidden files and `sample_id` as its row ID. **We re-run exactly these two cells with `TEST_FILE` and `AUCTION_FILE` set to the hidden test and auction files. Restart the kernel and confirm that these two cells run on their own.**

In [ ]:
import gzip, pickle
import numpy as np
import pandas as pd

MODEL_FILE   = "team_model.pkl"
TEST_FILE    = "sample_features.csv"   # we use the hidden test file
AUCTION_FILE = "sample_features.csv"   # we use the hidden auction file

with gzip.open(MODEL_FILE, "rb") as f:
    saved_with = pickle.load(f)
    saved = pickle.load(f)
predict_revenue, auction_sheet = saved["predict_revenue"], saved["auction_sheet"]
print(f"Loaded {MODEL_FILE} (saved with Python {saved_with['python']}, scikit-learn {saved_with['scikit-learn']})")

In [ ]:
ID_COLUMNS = ["test_id", "auction_id", "sample_id"]

def read_films(features_file):
    films = pd.read_csv(features_file)
    id_col = next(column for column in ID_COLUMNS if column in films.columns)
    return films[[id_col]], films.drop(columns=id_col)

# predictions.csv: one positive revenue forecast per test film (scored)
ids, films = read_films(TEST_FILE)
forecast = np.asarray(predict_revenue(films), dtype=float)
if forecast.ndim == 2 and forecast.shape[1] == 1:
    forecast = forecast[:, 0]
assert forecast.shape == (len(films),), f"expected one forecast per film, got shape {forecast.shape}"
assert np.isfinite(forecast).all() and (forecast > 0).all(), "every forecast must be a positive number"
ids.assign(predicted_revenue=forecast).to_csv("predictions.csv", index=False)
print(f"✅ predictions.csv: {len(forecast)} rows, median {np.median(forecast):,.0f} USD")

# valuations.csv: your auction sheet, with any columns you like (not scored)
ids, films = read_films(AUCTION_FILE)
sheet = pd.DataFrame(auction_sheet(films)).reset_index(drop=True)
assert len(sheet) == len(films), f"{len(sheet)} auction sheet rows for {len(films)} films"
assert sheet.columns.is_unique, "auction sheet column names must be unique"
assert not set(ID_COLUMNS) & set(sheet.columns), "section 6 adds the ID column; do not return it"
pd.concat([ids, sheet], axis=1).to_csv("valuations.csv", index=False)
print(f"✅ valuations.csv: {len(sheet)} rows, columns: {', '.join(map(str, sheet.columns))}")